# Fine Tuned CatBoost

## Importing Packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install catboost
from catboost import CatBoostClassifier, Pool

## Importing data

In [ ]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }
    
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

## Setting up our columns

In [ ]:
target_col = 'cost_category'
id_col = 'Tour_ID'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

## Advanced Feature Engineering Pipeline

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # 1. Clean typos and missing values
    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})
    
    df['travel_with'] = df['travel_with'].fillna('Alone')
    df['total_female'] = df['total_female'].fillna(0)
    df['total_male'] = df['total_male'].fillna(0)
    df['most_impressing'] = df.get('most_impressing', pd.Series()).fillna('No Answer')

    # 2. Aggregations & Group Metrics
    df['total_people'] = df['total_female'] + df['total_male']
    df['total_people'] = df['total_people'].apply(lambda x: 1 if x == 0 else x) # prevent div by zero
    
    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    
    # Ratios
    df['mainland_ratio'] = df['night_mainland'] / (df['total_nights'] + 1e-5)
    df['zanzibar_ratio'] = df['night_zanzibar'] / (df['total_nights'] + 1e-5)
    df['female_ratio'] = df['total_female'] / df['total_people']
    df['nights_per_person'] = df['total_nights'] / df['total_people']
    
    # 3. Package Inclusions Breakdown
    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    
    for p_col in package_cols:
        if p_col in df.columns:
            df[p_col + '_binary'] = (df[p_col] == 'Yes').astype(int)
            
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    df['is_full_package'] = (df['package_count'] == len(package_cols)).astype(int)
    df['has_no_package'] = (df['package_count'] == 0).astype(int)
    
    # 4. Country / Region Frequency Encoding
    country_counts = df['country'].value_counts()
    df['country_freq'] = df['country'].map(country_counts)

    return df

train_df = engineer_features(train)
test_df = engineer_features(test)

## Categorical Feature Specification

In [ ]:
cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]

# Ensure categorical variables are string formatted for CatBoost
for col in cat_features:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

X = train_df[features]
y = train_df['target']
X_test = test_df[features]

## Stratified 5-Fold Cross-Validation with LightGBM

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

test_pool = Pool(X_test, cat_features=cat_features)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    
    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.035,
        depth=6,
        l2_leaf_reg=4,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=42,
        verbose=300
    )
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=120,
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict_proba(val_pool)
    test_preds += model.predict_proba(test_pool) / skf.n_splits


cv_log_loss = log_loss(y, oof_preds)
print(f"\n==========================================")
print(f"Overall OOF Log Loss: {cv_log_loss:.5f}")
print(f"==========================================")

## Fine tuned catboost model Submission

In [ ]:
submission = pd.DataFrame(test_preds, columns=[idx_to_class[i] for i in range(len(target_classes))])
submission.insert(0, id_col, test_df[id_col])

# Align strictly with sample submission format
submission = submission[['Tour_ID', 'High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']]
submission.to_csv('advanced_catboost_submission.csv', index=False)